# GSE262619 免疫老化 遺伝子発現 解析
## Young(22-31) → Middle(57-64) → Old(75-88) の2遷移比較

**このコホートの制約（必ず結果解釈時に明記すること）**：年齢は22-31/57-64/75-88の3クラスターのみで、
32-56歳・65-74歳の区間にサンプルが存在しない。したがってHRV由来のCHIピーク(44.9歳)・α2ピーク(67.1歳)を
このコホート単独で直接検証することはできない。本解析は、Young→Middle（前半の遷移、CHI/α1の年代を含む区間）と
Middle→Old（後半の遷移、α2の年代を含む区間）の**2群比較**として設計し、「一致した」「証明した」ではなく、
「時間的に対応する」「整合的である」という表現で結果を記述する。


In [ ]:
# ============================================================
# Step 1: 環境セットアップ
# ============================================================
!pip install gseapy==1.1.3 -q

import os
import re
import gzip
import glob
import shutil
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# --- Google Driveのマウント (フォルダが残っていて失敗するケース・既にマウント済みのケースの両方に対応) ---
from google.colab import drive

if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Driveは既にマウント済みです。')

# --- ファイルをDrive全体(MyDrive + 共有ドライブ)から自動検索 ---
# 完全一致ではなく「キーワードを全て含むファイル名」で探すことで、
# Drive側でのリネーム(例: 末尾に" (1)"が付く等)や大文字/小文字の違いにも対応する。
def find_file(keywords, search_root='/content/drive'):
    all_files = glob.glob(os.path.join(search_root, '**', '*'), recursive=True)
    all_files = [f for f in all_files if os.path.isfile(f)]
    candidates = [f for f in all_files
                  if all(kw.lower() in os.path.basename(f).lower() for kw in keywords)]
    if not candidates:
        print(f'"{" + ".join(keywords)}" を含むファイルが見つかりませんでした。')
        gse_related = [f for f in all_files if 'gse262619' in os.path.basename(f).lower()]
        if gse_related:
            print('"GSE262619"を含むファイルは以下が見つかっています(名前を確認してください):')
            for f in gse_related:
                print('  -', f)
        else:
            print('"GSE262619"を含むファイルがDrive上に1つも見つかりませんでした。')
            print('考えられる原因:')
            print('  1. アップロードがまだ完了していない/別のGoogleアカウントのDriveにアップロードした')
            print('  2. 「共有アイテム」に入れただけで、自分のMyDriveに追加(ショートカット等)していない')
            print('  3. アップロード先を勘違いしている(別フォルダ/別Drive)')
            print('参考: Drive直下のフォルダ一覧:')
            top_level = glob.glob('/content/drive/MyDrive/*')
            for t in top_level[:30]:
                print('  -', t)
        raise FileNotFoundError(f'{keywords} に一致するファイルが見つかりません。上のログを確認してください。')
    if len(candidates) > 1:
        print(f'注意: 複数のファイルが見つかりました。最初のものを使います:')
        for c in candidates:
            print('  -', c)
    return candidates[0]

SERIES_MATRIX_PATH = find_file(['GSE262619', 'series_matrix'])
RAW_COUNT_PATH      = find_file(['GSE262619', 'Raw_count'])
print('SERIES_MATRIX_PATH:', SERIES_MATRIX_PATH)
print('RAW_COUNT_PATH     :', RAW_COUNT_PATH)

OUTPUT_DIR = '/content/drive/MyDrive/GSE262619_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Step 2: series matrixからメタデータを頑健にパースする

`age (years): N` を含む `!Sample_characteristics_ch1` 行から年齢を抽出する。
性別はGEOのcharacteristicsフィールドに存在しないため、`!Sample_title`（例: "30y_F1"）の
`_F`/`_M` パターンから復元する。ハードコードではなくファイルから都度パースすることで、
将来データが更新された場合や別のGSEに流用する場合の再現性を確保する。


In [ ]:
# ============================================================
# Step 2: series matrixからメタデータをパース
# ============================================================
def parse_series_matrix_metadata(path):
    with gzip.open(path, 'rt') as f:
        lines = f.readlines()

    title_line = None
    accession_line = None
    char_lines = []
    for line in lines:
        if line.startswith('!Sample_title'):
            title_line = line
        elif line.startswith('!Sample_geo_accession'):
            accession_line = line
        elif line.startswith('!Sample_characteristics_ch1'):
            char_lines.append(line)

    def split_tsv_quoted(line):
        return [x.strip().strip('"') for x in line.rstrip('\n').split('\t')[1:]]

    titles = split_tsv_quoted(title_line)
    accessions = split_tsv_quoted(accession_line) if accession_line else [None] * len(titles)

    age = None
    for cl in char_lines:
        vals = split_tsv_quoted(cl)
        if vals and vals[0].lower().startswith('age'):
            age = [int(re.search(r'(\d+)', v).group(1)) for v in vals]
            break
    if age is None:
        raise ValueError('age (years) の行が見つかりませんでした。characteristics_ch1の内容を確認してください。')

    sex = []
    for t in titles:
        m = re.search(r'_([FM])\d+$', t)
        sex.append(m.group(1) if m else None)

    meta = pd.DataFrame({'title': titles, 'geo_accession': accessions, 'age': age, 'sex': sex})
    meta['group'] = pd.cut(meta['age'], bins=[0, 35, 70, 200], labels=['Young', 'Middle', 'Old'])
    return meta

meta = parse_series_matrix_metadata(SERIES_MATRIX_PATH)
print(meta['group'].value_counts())
print(meta.groupby('group')['age'].agg(['min', 'max', 'count']))
meta.to_csv(os.path.join(OUTPUT_DIR, 'sample_metadata.csv'), index=False)
meta.head()


## Step 3: raw countの読み込み・遺伝子シンボルへの集約・正規化

このデータはStringTie由来で、`gene_id`の多くが`MSTRG.xxxxx`形式の新規アセンブル遺伝子座であり、
標準遺伝子（`ENSG...`）は172,043行中5,729行のみ。ただし`gene_name`列には`MSTRG`行にも
既知遺伝子シンボルが付与されている場合が多く（例: MSTRG.10 → PLCXD1）、パスウェイ解析に使うのは
`gene_name`が適切。1つの`gene_name`に複数の`gene_id`が対応する場合があるため（遺伝子座の分断、
最大463個/遺伝子）、**raw countの段階で`gene_name`ごとに合算してから**フィルタリング・正規化する
（正規化後の値を後から合算するのは統計的に不適切）。


In [ ]:
# ============================================================
# Step 3: raw count読み込み → gene_name集約 → フィルタ → 正規化 (median-of-ratios)
# ============================================================
raw = pd.read_csv(RAW_COUNT_PATH, sep='\t')
raw = raw.set_index('gene_id')
gene_name_map = raw['gene_name']
count_mat = raw.drop(columns=['gene_name'])

samples = ['count.G' + t for t in meta['title']]
assert list(count_mat.columns) == samples, 'サンプル列の並びがmetaと一致していません'
count_mat = count_mat[samples]

# --- gene_nameごとにraw countを合算 ---
df = count_mat.copy()
df['gene_name'] = gene_name_map
df = df.dropna(subset=['gene_name'])
collapsed = df.groupby('gene_name')[samples].sum()
print(f'collapse前: {count_mat.shape[0]:,} 行 (gene_id) -> collapse後: {collapsed.shape[0]:,} 行 (gene_name)')

# --- フィルタ: いずれかの群で70%以上のサンプルで発現 ---
group_masks = {g: [c for c, gg in zip(samples, meta['group']) if gg == g] for g in ['Young', 'Middle', 'Old']}
def frac_nonzero(d, cols):
    return (d[cols] > 0).mean(axis=1)
frac_y = frac_nonzero(collapsed, group_masks['Young'])
frac_m = frac_nonzero(collapsed, group_masks['Middle'])
frac_o = frac_nonzero(collapsed, group_masks['Old'])
keep_mask = (frac_y >= 0.7) | (frac_m >= 0.7) | (frac_o >= 0.7)
filt = collapsed.loc[keep_mask]
print(f'発現フィルタ後: {filt.shape[0]:,} 遺伝子')

# --- DESeq2式 median-of-ratios 正規化 ---
log_filt = np.log(filt.replace(0, np.nan))
log_geomean = log_filt.mean(axis=1, skipna=False)
valid_genes = log_geomean.notna()
log_ratio = log_filt.loc[valid_genes].sub(log_geomean.loc[valid_genes], axis=0)
size_factors = np.exp(log_ratio.median(axis=0))
norm = filt.div(size_factors, axis=1)
log2norm = np.log2(norm + 1)

print(f'サイズファクター計算に使用した遺伝子数(全サンプルでcount>0): {valid_genes.sum():,}')
print(size_factors.describe())

log2norm.to_pickle(os.path.join(OUTPUT_DIR, 'log2norm.pkl'))
size_factors.to_csv(os.path.join(OUTPUT_DIR, 'size_factors.csv'))


## Step 4: QC (PCA)

上位2000変動遺伝子でPCAを行う。**注意**：この解析では性別(PC2、XIST等の性染色体遺伝子の影響で
分散が非常に大きい)が最も強い変動軸になりやすく、年齢(PC1)がそれより弱く現れることがある。
これは異常ではなく血球トランスクリプトームでは典型的な現象だが、性別を無視して群間比較すると
交絡するため、Step 5のDE検定では必ず性別で調整する。


In [ ]:
# ============================================================
# Step 4: QC - PCA (上位変動遺伝子, 性別/年齢群での色分け)
# ============================================================
var = log2norm.var(axis=1)
top = var.sort_values(ascending=False).head(2000).index
X = log2norm.loc[top].T.values
X = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
pc = U[:, :2] * S[:2]
var_explained = (S**2) / (S**2).sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor='#FFFFFF')
GROUP_COLORS = {'Young': '#3A6EA5', 'Middle': '#5A9367', 'Old': '#D9534F'}
SEX_COLORS = {'F': '#D9534F', 'M': '#3A6EA5'}

for ax, colorby, cmap in [(axes[0], meta['group'].values, GROUP_COLORS), (axes[1], meta['sex'].values, SEX_COLORS)]:
    ax.set_facecolor('#FFFFFF')
    for cat, color in cmap.items():
        mask = colorby == cat
        ax.scatter(pc[mask, 0], pc[mask, 1], color=color, label=cat, s=45, alpha=0.85, edgecolor='#333333', linewidth=0.4)
    ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)
    for spine in ax.spines.values():
        spine.set_edgecolor('#CCCCCC')

axes[0].set_title('PCA colored by age group')
axes[1].set_title('PCA colored by sex')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'qc_pca.png'), dpi=200, facecolor='#FFFFFF')
plt.show()


## Step 5: 差次的発現解析 (Young vs Middle, Middle vs Old, 性別調整)

各遺伝子について `log2(expr) ~ group + sex` の線形モデルをベクトル化して一括fit
(全遺伝子ループなしでnumpyの行列演算のみで実行、DESeq2/limmaがない環境でも実行可能)。
t統計量をGSEAのランキング指標として用いる (符号=方向, 大きさ=群間差の確からしさ)。


In [ ]:
# ============================================================
# Step 5: DE検定 (性別調整済み線形モデル, ベクトル化)
# ============================================================
def de_test(log2norm, meta, samples, group_a, group_b):
    mask = meta['group'].isin([group_a, group_b]).values
    sub_samples = [s for s, m in zip(samples, mask) if m]
    sub_meta = meta.loc[mask].reset_index(drop=True)
    Y = log2norm[sub_samples].values.T
    n = Y.shape[0]
    group_dummy = (sub_meta['group'] == group_b).astype(float).values
    sex_dummy = (sub_meta['sex'] == 'M').astype(float).values
    X = np.column_stack([np.ones(n), group_dummy, sex_dummy])
    k = X.shape[1]
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ Y
    resid = Y - X @ beta
    dof = n - k
    sigma2 = (resid**2).sum(axis=0) / dof
    se_b1 = np.sqrt(sigma2 * XtX_inv[1, 1])
    t_stat = beta[1] / se_b1
    p_val = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=dof))
    order = np.argsort(p_val)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(p_val) + 1)
    m = len(p_val)
    fdr = p_val * m / ranks
    fdr_sorted = np.minimum.accumulate(fdr[order][::-1])[::-1]
    fdr_final = np.empty_like(fdr)
    fdr_final[order] = fdr_sorted
    fdr_final = np.clip(fdr_final, 0, 1)
    return pd.DataFrame({'gene_name': log2norm.index, 'log2FC': beta[1], 't': t_stat, 'p': p_val, 'padj': fdr_final})

res_ym = de_test(log2norm, meta, samples, 'Young', 'Middle')
res_mo = de_test(log2norm, meta, samples, 'Middle', 'Old')

print(f'Young->Middle: FDR<0.05 = {(res_ym["padj"]<0.05).sum()} / {len(res_ym)}  (nominal p<0.01 = {(res_ym["p"]<0.01).sum()})')
print(f'Middle->Old:   FDR<0.05 = {(res_mo["padj"]<0.05).sum()} / {len(res_mo)}  (nominal p<0.01 = {(res_mo["p"]<0.01).sum()})')
print()
print('--- 遷移の大きさの比較 (この解析の主要な予備知見) ---')
print(f'Middle->Oldの方がYoung->Middleより{(res_mo["p"]<0.01).sum() / max((res_ym["p"]<0.01).sum(),1):.1f}倍多くの遺伝子で'
      f'nominal p<0.01の変化を示した。免疫トランスクリプトームの再編成は、少なくともこのコホートでは'
      f'後半の遷移(60→80歳, α2の年代を含む)に偏っている可能性を示唆する。')

res_ym.to_csv(os.path.join(OUTPUT_DIR, 'DE_young_vs_middle.csv'), index=False)
res_mo.to_csv(os.path.join(OUTPUT_DIR, 'DE_middle_vs_old.csv'), index=False)


## Step 6: GSEA (prerank, 両遷移で実施)

t統計量でランキングしたprerank GSEAを、Reactome/KEGG/GO-BP/Hallmarkの4ライブラリに対して実行する。
インターネット接続が必要（Enrichrのライブラリをダウンロードする）。


In [ ]:
# ============================================================
# Step 6: GSEA (prerank)
# ============================================================
import gseapy as gp

GENE_SETS = ['Reactome_2022', 'KEGG_2021_Human', 'GO_Biological_Process_2023', 'MSigDB_Hallmark_2020']

def run_prerank(res_df, label):
    rnk = res_df[['gene_name', 't']].dropna().sort_values('t', ascending=False)
    rnk = rnk.drop_duplicates(subset='gene_name', keep='first')  # 重複遺伝子名は念のため1つに
    pre_res = gp.prerank(
        rnk=rnk, gene_sets=GENE_SETS, min_size=10, max_size=500,
        permutation_num=1000, outdir=None, seed=42, threads=4, no_plot=True,
    )
    res2d = pre_res.res2d.copy()
    res2d['comparison'] = label
    print(f'{label}: {res2d.shape[0]} pathways tested across {len(GENE_SETS)} libraries')
    print('columns:', list(res2d.columns))
    return res2d

gsea_ym = run_prerank(res_ym, 'Young_vs_Middle')
gsea_mo = run_prerank(res_mo, 'Middle_vs_Old')

gsea_ym.to_csv(os.path.join(OUTPUT_DIR, 'GSEA_young_vs_middle.csv'), index=False)
gsea_mo.to_csv(os.path.join(OUTPUT_DIR, 'GSEA_middle_vs_old.csv'), index=False)


## Step 7: テーマ別（炎症・酸化ストレス・ミトコンドリア・細胞老化）パスウェイの抽出と比較

事後的に都合の良いパスウェイを拾わないよう、4テーマのキーワードは解析前に固定する。
両遷移(Young→Middle, Middle→Old)でヒットしたパスウェイを名前で突き合わせ、NES・FDRを並べて比較する。


In [ ]:
# ============================================================
# Step 7: テーマタグ付け + 早期/後期遷移の比較テーブル
# ============================================================
THEME_KEYWORDS = {
    'inflammation':    [r'inflamm', r'\bnf.?kb\b', r'\btnf\b', r'interleukin', r'\bil-?\d', r'cytokine',
                         r'interferon', r'complement', r'toll.?like', r'chemokine'],
    'oxidative_stress': [r'oxidative', r'reactive oxygen', r'\bros\b', r'antioxidant', r'glutathione', r'nrf2', r'nfe2l2'],
    'mitochondrial':    [r'mitochondri', r'oxidative phosphorylation', r'electron transport', r'respiratory chain', r'\batp synthesis'],
    'senescence':       [r'senescen', r'\bsasp\b', r'cell cycle arrest', r'\bp16\b', r'\bp21\b', r'telomere'],
}

def tag_themes(term):
    term_low = str(term).lower()
    tags = [theme for theme, patterns in THEME_KEYWORDS.items()
            if any(re.search(p, term_low) for p in patterns)]
    return ';'.join(tags) if tags else ''

for df_ in [gsea_ym, gsea_mo]:
    df_['theme'] = df_['Term'].apply(tag_themes)

themed_ym = gsea_ym[gsea_ym['theme'] != ''].copy()
themed_mo = gsea_mo[gsea_mo['theme'] != ''].copy()
print(f'テーマに合致したパスウェイ数: Young->Middle = {len(themed_ym)}, Middle->Old = {len(themed_mo)}')

# --- 両遷移をTermで突き合わせ ---
cols = ['Term', 'theme', 'NES', 'FDR q-val']
merged = pd.merge(
    themed_ym[cols].rename(columns={'NES': 'NES_young_middle', 'FDR q-val': 'FDR_young_middle'}),
    themed_mo[cols].rename(columns={'NES': 'NES_middle_old', 'FDR q-val': 'FDR_middle_old'}),
    on=['Term', 'theme'], how='outer'
)
merged = merged.sort_values(by=['theme', 'Term'])
merged.to_csv(os.path.join(OUTPUT_DIR, 'theme_pathway_comparison.csv'), index=False)

for theme in THEME_KEYWORDS:
    sub = merged[merged['theme'].str.contains(theme, na=False)]
    print(f'\n=== {theme} ({len(sub)} pathways) ===')
    print(sub[['Term', 'NES_young_middle', 'FDR_young_middle', 'NES_middle_old', 'FDR_middle_old']].to_string(index=False))


## Step 8: テーマ別サマリー可視化

各テーマについて、有意(FDR<0.25、GSEAの慣例的閾値)なパスウェイ数をYoung→MiddleとMiddle→Oldで比較する。


In [ ]:
# ============================================================
# Step 8: テーマ別 有意パスウェイ数の比較(棒グラフ)
# ============================================================
FDR_THRESH = 0.25  # GSEAの慣例的な閾値 (Subramanian et al. 2005)

summary_rows = []
for theme in THEME_KEYWORDS:
    n_ym = ((merged['theme'].str.contains(theme, na=False)) & (merged['FDR_young_middle'] < FDR_THRESH)).sum()
    n_mo = ((merged['theme'].str.contains(theme, na=False)) & (merged['FDR_middle_old'] < FDR_THRESH)).sum()
    summary_rows.append({'theme': theme, 'Young_to_Middle': n_ym, 'Middle_to_Old': n_mo})
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

fig, ax = plt.subplots(figsize=(8, 5), facecolor='#FFFFFF')
ax.set_facecolor('#FFFFFF')
x = np.arange(len(summary_df))
w = 0.35
ax.bar(x - w/2, summary_df['Young_to_Middle'], width=w, color='#3A6EA5', label='Young to Middle\n(~27-60y, includes CHI/alpha1 age range)')
ax.bar(x + w/2, summary_df['Middle_to_Old'], width=w, color='#D9534F', label='Middle to Old\n(~60-80y, includes alpha2 age range)')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['theme'], fontsize=10)
ax.set_ylabel(f'Significant pathways (FDR < {FDR_THRESH})')
ax.set_title('Theme-level pathway enrichment: early vs. late transition')
ax.legend(fontsize=8.5)
ax.grid(True, axis='y', alpha=0.15)
for spine in ax.spines.values():
    spine.set_edgecolor('#CCCCCC')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'theme_comparison_barplot_fixed.png'), dpi=300, facecolor='#FFFFFF')
plt.show()


## Step 9: まとめと解釈上の注意（必読）

- このコホートは32–56歳・65–74歳を含まないため、CHI(44.9歳)・α2(67.1歳)ピークの**直接検証はできない**。
- 主張してよいのは「Young→Middleの変化」と「Middle→Oldの変化」の相対的な大きさ・パスウェイ内容の比較のみ。
- 対応づけの表現は必ず時間的対応・整合性の言葉を使う（「一致した」「証明した」は使わない）。
  - 良い例：「Middle→Oldで観察された炎症関連パスウェイの変化は、HRV解析でα2ピーク以降に観察される
    生理学的変化と時間的に対応する可能性がある。」
- n=10〜14/群と小標本のため、遺伝子レベルのFDR<0.05はごく少数（探索的解析として妥当な範囲）。
  GSEAはパスウェイ単位に集約することでこの限界をある程度補っているが、それでも探索(discovery)コホートである。
- 次のステップとして、40〜70歳を連続的に含む別コホート(bulk RNA-seqまたはsingle-cell)での再現確認を推奨する。
